In [ ]:
from dotenv import load_dotenv
import os
import anthropic

In [ ]:
load_dotenv()  # reads .env from the current directory

In [ ]:
client = anthropic.Anthropic(
    base_url="https://claude.vocareum.com",
    api_key=os.environ["ANTHROPIC_API_KEY"],
)

In [ ]:
response = client.messages.create(
    model="claude-sonnet-4-5-20250929",
    max_tokens=1024,
    messages=[{"role": "user", "content": "Hello!"}],
)

print(response.content[0].text)

In [ ]:
LARGE_CONTEXT = """<paste a long document, codebase, or instructions here —
must be at least ~1024 tokens to actually cache (4096 on Opus 4.5 and Haiku 4.5)>"""

def ask(question: str):
    response = client.messages.create(
        model="claude-sonnet-4-5-20250929",
        max_tokens=1024,
        system=[
            {
                "type": "text",
                "text": LARGE_CONTEXT,
                "cache_control": {"type": "ephemeral"},  # cache everything up to here
            }
        ],
        messages=[{"role": "user", "content": question}],
    )

    # Verify caching worked
    print(f"cache_creation_input_tokens: {response.usage.cache_creation_input_tokens}")
    print(f"cache_read_input_tokens:     {response.usage.cache_read_input_tokens}")
    print(f"input_tokens (uncached):     {response.usage.input_tokens}")

    for block in response.content:
        if block.type == "text":
            print(block.text)

ask("Summarize the key points.")    # 1st call: writes cache
ask("What's the main argument?")    # 2nd call: reads cache (much cheaper)

### Available models on Vocareum
- claude-haiku-4-5-20251001
- claude-sonnet-4-5-20250929
- claude-opus-4-5-20251101

### Recommended defaults:

- General use / best capability: `claude-opus-4-5-20251101`
- Best speed/cost balance: `claude-sonnet-4-5-20250929`
- Cheapest/fastest: `claude-haiku-4-5-20251001`